In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [llama_stack_client]lama_stack_client]


In [2]:
import os
import sys
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
import pandas as pd
import logging
import requests
from io import BytesIO

In [34]:
sys.path.append('..')
# Load environment variables from .env file
load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel("INFO")

# Initialize the Llama Stack client
client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_SERVER_URL", "http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321")
)

file_path = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"
url = "https://www.openshift.guide/openshift-guide-screen.pdf"
vector_db_skus_name = "skus_rh_vector_db"
vector_db_ocp_name = "ocp_rh_vector_db"

logger.info("Connected to Llama Stack server")

INFO:__main__:Connected to Llama Stack server


In [51]:
for m in client.models.list():
    print(f"Model: {m}")
    #if m.id == "sentence-transformers/nomic-ai/nomic-embed-text-v1.5":
        #client.models.unregister(
        #    model_id=m.id
        #)

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


Model: Model(id='llama-3-2-3b/llama-3-2-3b', created=1777482191, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'llama-3-2-3b', 'provider_resource_id': 'llama-3-2-3b'}, object='model')
Model: Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1777482191, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model')
Model: Model(id='vllm-inference/redhataillama-31-8b-instruct', created=1777482191, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'vllm-inference', 'provider_resource_id': 'redhataillama-31-8b-instruct'}, object='model')
Model: Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1777482191, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provid

In [74]:
vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

for vector_store in vector_stores:
    client.vector_stores.delete(
        vector_store_id=vector_store.id
    )
    print(f"Vector store: {vector_store.name} deleted")

print("All Vector stores deleted")

vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id=None, object='list', first_id=None)
All Vector stores deleted
Vector stores SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id=None, object='list', first_id=None)


In [75]:
# Upload a document
file = client.files.create(
    file=open(file_path, "rb"),
    purpose="assistants",
)
print(f"Uploaded: {file.id}")

# Create a vector store and index the file
vector_store_skus = client.vector_stores.create(
    name=vector_db_skus_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

client.vector_stores.files.create(
    vector_store_id=vector_store_skus.id,
    file_id=file.id,
    attributes={
        "document_id": "Subscriptions.csv",
        "source": file_path
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file.id} loaded into Vector store SKUs with ID: {vector_store_skus.id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Uploaded: file-2af2af92080d40de8fb46f680d1a765e


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_29e682dd-044a-46fc-85c5-11bf6c096a48/files "HTTP/1.1 200 OK"


File file-2af2af92080d40de8fb46f680d1a765e loaded into Vector store SKUs with ID: vs_29e682dd-044a-46fc-85c5-11bf6c096a48


In [78]:
response = requests.get(url)
file_buffer = BytesIO(response.content)
file_buffer.name = "openshift-guide-screen.pdf"

# Upload a document
file_url = client.files.create(
    file=file_buffer,
    purpose="assistants",
)
print(f"Uploaded: {file_url.id}")

# Create a vector store and index the file
vector_store_ocp = client.vector_stores.create(
    name=vector_db_ocp_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

client.vector_stores.files.create(
    vector_store_id=vector_store_ocp.id,
    file_id=file_url.id,
    attributes={
        "document_id": file_buffer.name,
        "source": url
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file_url.id} loaded into Vector store OCP with ID: {vector_store_ocp.id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Uploaded: file-eb0bfff1d07944d496457a77b31a6236


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_5c9680d5-1ad0-4203-89a0-f8996760c8ae/files "HTTP/1.1 200 OK"


File file-eb0bfff1d07944d496457a77b31a6236 loaded into Vector store OCP with ID: vs_5c9680d5-1ad0-4203-89a0-f8996760c8ae


In [ ]:
vector_stores = client.vector_stores.list()

print(f"Vector stores created {vector_stores}")

In [80]:
query = "List of Red Hat OpenShift Container Platform SKU"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_skus.id],
    }],
)

logger.info(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from skus_rh_vector_db - Result: 
Based on the search results, the list of Red Hat OpenShift Container Platform SKU includes:

* Red Hat Connectivity Link, 5M Gateway Requests per day, Premium
* Red Hat Connectivity Link, 10M Gateway Requests per day, Premium
* Red Hat Support for IBM Cloud Pak (Red Hat OpenShift only), Premium (1 Core)
* Red Hat OpenShift Container Platform (Bare Metal Node), Premium (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node), Standard (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node) Extended Update Support Long-Life Add-On - Term 1 (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node) Extended Update Support Long-Life Add-On - Term 2 (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Data Found

In [81]:
query = "What is Red Hat OpenShift?"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_ocp.id],
    }],
)

logger.info(f"RAG Query from {vector_db_ocp_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from ocp_rh_vector_db - Result: 
Red Hat OpenShift is an enterprise-class platform built upon Kubernetes and provides a full DevOps product ready to use. It is designed with high availability and security in mind and integrates a whole host of DevOps tools in a single package. Among its features, we can find built-in user and group management, tighter security requirements for containers, more robust namespace isolation through projects, an integrated visual management console, an embedded container registry, a CI/CD pipeline system, a built-in Cloud Native application store featuring ready-to-use applications bundled as Kubernetes Operators, and integrated management and logging features. Red Hat OpenShift clusters feature more robust security defaults than stock Kubernetes and can also run in high-availability mode, ensuri